# MaizeGuard Final Training Notebook

This notebook trains the final MaizeGuard image classifier and exports the files required by the backend.

Classes: `good`, `broken`, `impurity`, and `mold_risk`. The deployed application may return `needs_review` when image quality is poor or when `broken` and `mold_risk` are too close.

Main outputs in `/kaggle/working/maizeguard_v9_label_audit/outputs/`:

- `maizeguard_public_best_model.pt`
- `maizeguard_model_metadata.json`
- `class_names.json`
- `model_metrics_summary.csv`
- `classification_report_raw_argmax.csv`
- `broken_mold_review_template.csv`
- `broken_mold_audit_pages/`

Run once to generate the review template and audit pages. If any broken/mold images are mixed or unclear, set their decision to `exclude` in `broken_mold_review_decisions.csv`, upload that file to Kaggle, and run the notebook again from the first cell.


## Method

The notebook uses original public class-labeled images, conservative color-preserving augmentation, MobileNetV3 transfer learning, gentle class-weighted cross-entropy, early stopping, and validation-only checkpoint selection. It does not tune on the test set.


In [ ]:
# ============================================================
# 0. Install/import dependencies
# ============================================================

import sys
import subprocess
import importlib.util
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

ensure_package("timm", "timm")

import os
import json
import math
import time
import shutil
import random
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def choose_safe_device():
    """Use CUDA only when the Kaggle GPU is supported by the installed PyTorch build."""
    if not torch.cuda.is_available():
        print("[INFO] CUDA is not available. Using CPU.")
        return torch.device("cpu"), False

    try:
        gpu_name = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        print(f"[INFO] Detected GPU: {gpu_name} | CUDA capability: sm_{major}{minor}")

        # Kaggle's current PyTorch builds commonly exclude Tesla P100 / sm_60 kernels.
        if major < 7:
            print("[WARN] This GPU is older than sm_70. Falling back to CPU to avoid cudaErrorNoKernelImageForDevice.")
            print("[TIP] Choose T4, V100, A100, or another newer GPU when available.")
            return torch.device("cpu"), False

        test = torch.randn(1, 3, 8, 8, device="cuda")
        conv = torch.nn.Conv2d(3, 4, 3).to("cuda")
        _ = conv(test)
        torch.cuda.synchronize()
        print("[INFO] CUDA test passed. Using GPU.")
        return torch.device("cuda"), True
    except Exception as error:
        print("[WARN] CUDA failed the safety test. Falling back to CPU:", repr(error))
        return torch.device("cpu"), False


DEVICE, CUDA_USABLE = choose_safe_device()
if CUDA_USABLE:
    torch.cuda.manual_seed_all(SEED)
print("Selected device:", DEVICE)

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/maizeguard_v9_label_audit")
RAW_PUBLIC_ROOT = WORK_ROOT / "raw_public_sources"
READY_ROOT = WORK_ROOT / "maizeguard_ready_dataset"
OUTPUT_DIR = WORK_ROOT / "outputs"
EXTERNAL_TEST_DIR = WORK_ROOT / "real_external_test"
AUDIT_DIR = OUTPUT_DIR / "broken_mold_audit_pages"

AUTO_DOWNLOAD_PUBLIC_DATASETS_IF_MISSING = True
CK_CNN_GIT_URL = "https://github.com/vision-cidis/CK-CNN.git"
CK_CNN_ZIP_URLS = [
    "https://github.com/vision-cidis/CK-CNN/archive/refs/heads/master.zip",
    "https://github.com/vision-cidis/CK-CNN/archive/refs/heads/main.zip",
]
for folder in [WORK_ROOT, RAW_PUBLIC_ROOT, READY_ROOT, OUTPUT_DIR, EXTERNAL_TEST_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CLASSES = ["good", "broken", "impurity", "mold_risk"]
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASSES)}
IDX_TO_CLASS = {index: name for name, index in CLASS_TO_IDX.items()}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_PER_CLASS = {label: 3500 for label in CLASSES}

# A review CSV can be uploaded in a Kaggle dataset. It must contain review_id or source_path,
# plus a decision column. Valid decisions: keep, exclude, broken, mold_risk.
REVIEW_FILE_NAME = "broken_mold_review_decisions.csv"
APPLY_HUMAN_REVIEW_DECISIONS = True

MODEL_NAME = "mobilenetv3_large_100"
IMG_SIZE = 320
BATCH_SIZE = 24 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

# Conservative schedule: early stopping prevents training beyond the useful validation point.
HEAD_EPOCHS = 5 if DEVICE.type == "cuda" else 3
FINETUNE_EPOCHS = 15 if DEVICE.type == "cuda" else 10
EARLY_STOPPING_PATIENCE = 4
LR_HEAD = 4e-4
LR_FINETUNE = 1e-5
WEIGHT_DECAY = 3e-4
MAX_GRAD_NORM = 1.0

# No weighted sampler: it can repeatedly show the same small set of images.
# A gentle square-root class weight is used only in the training loss.
USE_CLASS_WEIGHTED_LOSS = True
CLASS_WEIGHT_POWER = 0.25

OFFICIAL_EVAL_MODE = "raw argmax, single image pass, untouched public holdout split"
NEEDS_REVIEW_CONFIDENCE_BELOW = 0.65
NEEDS_REVIEW_TOP2_MARGIN_BELOW = 0.15
BROKEN_MOLD_DEPLOYMENT_MARGIN_BELOW = 0.12
BROKEN_MOLD_DEPLOYMENT_MIN_PROBABILITY = 0.20
MIN_EXTERNAL_IMAGE_SIDE = 160
MIN_EXTERNAL_BLUR_SCORE = 12.0
PATCH_CONSENSUS_MINIMUM = 0.50

CONFIG = {
    "notebook_version": "V9_label_audit_conservative_training",
    "model_backbone": MODEL_NAME,
    "pretraining": "ImageNet pretrained via timm",
    "input_image_size": IMG_SIZE,
    "classes": CLASSES,
    "loss": "gentle_class_weighted_cross_entropy",
    "class_weight_power": CLASS_WEIGHT_POWER,
    "optimizer": "AdamW",
    "lr_head": LR_HEAD,
    "lr_finetune": LR_FINETUNE,
    "weight_decay": WEIGHT_DECAY,
    "official_evaluation": OFFICIAL_EVAL_MODE,
    "sampling_strategy": "shuffle_train_loader_no_weighted_sampler",
    "human_review_file_name": REVIEW_FILE_NAME,
    "selected_device": str(DEVICE),
    "cuda_usable": CUDA_USABLE,
    "deployment_safety_rule": {
        "needs_review_confidence_below": NEEDS_REVIEW_CONFIDENCE_BELOW,
        "needs_review_top2_margin_below": NEEDS_REVIEW_TOP2_MARGIN_BELOW,
        "broken_mold_probability_margin_below": BROKEN_MOLD_DEPLOYMENT_MARGIN_BELOW,
        "broken_mold_min_probability": BROKEN_MOLD_DEPLOYMENT_MIN_PROBABILITY,
        "patch_consensus_minimum": PATCH_CONSENSUS_MINIMUM,
    },
}
(OUTPUT_DIR / "training_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
CONFIG


## Dataset Preparation and Human Review

Images are mapped into the four MaizeGuard classes and split into train, validation, and test sets. The review file is optional, but it is the correct way to handle unclear `broken` versus `mold_risk` images. Valid review decisions are `keep`, `exclude`, `broken`, and `mold_risk`.


In [ ]:
# ============================================================
# 3. Review-file discovery and audit export
# ============================================================

VALID_REVIEW_DECISIONS = {"keep", "exclude", "broken", "mold_risk"}


def review_id_for_path(path_value: str) -> str:
    return hashlib.sha1(str(path_value).encode("utf-8")).hexdigest()[:16]


def find_review_csv():
    candidates = [OUTPUT_DIR / REVIEW_FILE_NAME, WORK_ROOT / REVIEW_FILE_NAME]
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(KAGGLE_INPUT_ROOT.rglob(REVIEW_FILE_NAME))

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def apply_human_review(records):
    """Apply an optional human review file before splitting.

    The file may identify rows using `review_id` or `source_path` and needs a
    `decision` column. Valid decisions are keep, exclude, broken, and mold_risk.
    """
    for row in records:
        row["review_id"] = review_id_for_path(row["source_path"])
        row["original_label"] = row["label"]
        row["review_decision"] = "not_reviewed"

    review_path = find_review_csv() if APPLY_HUMAN_REVIEW_DECISIONS else None
    if review_path is None:
        print("[INFO] No reviewed decision file found. Training will use the original public labels.")
        return records, pd.DataFrame()

    decisions_df = pd.read_csv(review_path)
    if "decision" not in decisions_df.columns:
        print(f"[WARN] Ignoring {review_path}: it does not contain a 'decision' column.")
        return records, decisions_df

    if "review_id" not in decisions_df.columns and "source_path" not in decisions_df.columns:
        print(f"[WARN] Ignoring {review_path}: include either 'review_id' or 'source_path' so rows can be matched.")
        return records, decisions_df

    decisions_df["decision"] = decisions_df["decision"].astype(str).str.strip().str.lower()
    invalid_mask = ~decisions_df["decision"].isin(VALID_REVIEW_DECISIONS)
    if invalid_mask.any():
        invalid_values = sorted(decisions_df.loc[invalid_mask, "decision"].dropna().unique().tolist())
        print(f"[WARN] Ignoring {int(invalid_mask.sum())} review row(s) with invalid decisions: {invalid_values}")
    decisions_df = decisions_df[~invalid_mask].copy()

    if decisions_df.empty:
        print(f"[WARN] No usable review decisions were found in {review_path}. Training will use original labels.")
        return records, decisions_df

    by_id = {}
    by_path = {}
    for _, row in decisions_df.iterrows():
        if pd.notna(row.get("review_id", None)):
            by_id[str(row["review_id"])] = str(row["decision"])
        if pd.notna(row.get("source_path", None)):
            by_path[str(row["source_path"])] = str(row["decision"])

    reviewed_records = []
    changed = 0
    excluded = 0
    for row in records:
        decision = by_id.get(row["review_id"], by_path.get(str(row["source_path"]), "keep"))
        row["review_decision"] = decision

        if decision == "exclude":
            excluded += 1
            continue
        if decision in {"broken", "mold_risk"} and decision != row["label"]:
            row["label"] = decision
            changed += 1
        reviewed_records.append(row)

    after_counts = Counter(row["label"] for row in reviewed_records)
    print(f"[INFO] Applied human review file: {review_path}")
    print(f"[INFO] Excluded {excluded} ambiguous rows; relabelled {changed} rows.")
    print("[INFO] Label counts after review:", dict(after_counts))
    return reviewed_records, decisions_df


def export_review_template(records):
    rows = []
    for row in records:
        if row["label"] not in {"broken", "mold_risk"}:
            continue
        rows.append({
            "review_id": row["review_id"],
            "source_path": str(row["source_path"]),
            "initial_label": row["label"],
            "decision": "keep",
            "reviewer_notes": "",
        })

    template = pd.DataFrame(rows).sort_values(["initial_label", "source_path"]).reset_index(drop=True)
    path = OUTPUT_DIR / "broken_mold_review_template.csv"
    template.to_csv(path, index=False)
    print("Saved review template:", path)
    return template


def export_audit_pages(template, images_per_page=20):
    """Create numbered contact sheets so human label review is practical."""
    if AUDIT_DIR.exists():
        shutil.rmtree(AUDIT_DIR)
    AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    if template.empty:
        print("[WARN] No broken/mold-risk rows were available for audit-page export.")
        return

    columns = 5
    rows_per_page = int(math.ceil(images_per_page / columns))
    page_count = 0

    for start in range(0, len(template), images_per_page):
        block = template.iloc[start:start + images_per_page].reset_index(drop=True)
        fig, axes = plt.subplots(rows_per_page, columns, figsize=(columns * 3.0, rows_per_page * 3.3))
        axes = np.asarray(axes).reshape(-1)
        for axis in axes:
            axis.axis("off")

        for axis, (_, row) in zip(axes, block.iterrows()):
            try:
                image = Image.open(row["source_path"]).convert("RGB")
                axis.imshow(image)
                axis.set_title(
                    f"{row['initial_label']}\nID: {row['review_id']}",
                    fontsize=8,
                )
            except Exception as error:
                axis.set_title(f"Unreadable\n{row['review_id']}", fontsize=8)

        fig.suptitle("Human label audit: broken vs mold_risk", fontsize=14)
        plt.tight_layout()
        page_path = AUDIT_DIR / f"broken_mold_audit_page_{page_count + 1:02d}.png"
        plt.savefig(page_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        page_count += 1

    print(f"Saved {page_count} audit page(s) to: {AUDIT_DIR}")



# ============================================================
# 2. Dataset discovery, label mapping, split, and preparation
# ============================================================

import zipfile
import urllib.request
import subprocess
import shutil
from collections import Counter


def normalize_text(path: Path) -> str:
    return str(path).lower().replace("\\", "/").replace("-", "_").replace(" ", "_")


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def is_blocked_non_training_asset(path: Path) -> bool:
    text = normalize_text(path)

    # Do NOT block folder names like "individual/good".
    # Only block non-training files/folders.
    blocked_words = [
        "/.git/", "/figs/", "/models/", "/code/",
        "mask", "masks", "segmentation", "annotation", "annotations",
        "json", "xml", "readme", "metadata",
        "groundtruth", "ground_truth", "csv", "txt", "plot", "chart"
    ]

    return any(word in text for word in blocked_words)


def safe_open_image(path: Path) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def file_hash(path: Path) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def list_input_folders(max_depth: int = 5):
    print("\n[DIAGNOSTIC] Existing folders under /kaggle/input and raw_public_sources:")
    roots = [KAGGLE_INPUT_ROOT, RAW_PUBLIC_ROOT, Path("/kaggle/working")]
    for root in roots:
        if not root.exists():
            print(f"  Missing: {root}")
            continue

        print(f"\n  Root: {root}")
        shown = 0
        for p in sorted(root.rglob("*")):
            if not p.is_dir():
                continue
            try:
                rel_depth = len(p.relative_to(root).parts)
            except Exception:
                rel_depth = 999
            if rel_depth <= max_depth:
                print("   ", p)
                shown += 1
            if shown >= 120:
                print("    ... stopped after 120 folders")
                break


def run_command(cmd, cwd=None):
    print("[CMD]", " ".join(map(str, cmd)))
    try:
        result = subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=600,
        )
        print(result.stdout[-3000:])
        return result.returncode == 0
    except Exception as e:
        print("[WARN] Command failed:", repr(e))
        return False


def download_zip(url: str, destination: Path) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    zip_path = destination.parent / f"{destination.name}.zip"

    try:
        print(f"[INFO] Downloading zip: {url}")
        urllib.request.urlretrieve(url, zip_path)

        tmp_dir = destination.parent / f"_{destination.name}_zip_extract"
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        tmp_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(tmp_dir)

        extracted_dirs = [p for p in tmp_dir.iterdir() if p.is_dir()]
        if not extracted_dirs:
            print("[WARN] ZIP extracted but no folder was found.")
            return False

        if destination.exists():
            shutil.rmtree(destination)

        shutil.move(str(extracted_dirs[0]), str(destination))
        shutil.rmtree(tmp_dir)
        zip_path.unlink(missing_ok=True)

        print(f"[INFO] ZIP extracted to: {destination}")
        return True
    except Exception as e:
        print(f"[WARN] Zip download failed: {url}\n       {repr(e)}")
        return False


def clone_or_download_repo(repo_url: str, zip_urls: list[str], destination: Path) -> bool:
    """Clone a repo, with GitHub zip fallback."""
    if destination.exists():
        print(f"[INFO] Repo folder already exists: {destination}")
        return True

    destination.parent.mkdir(parents=True, exist_ok=True)

    ok = run_command(["git", "clone", "--depth", "1", repo_url, str(destination)])
    if ok and destination.exists():
        print(f"[INFO] Cloned repo to: {destination}")
        return True

    print("[WARN] git clone failed. Trying ZIP fallback...")
    for url in zip_urls:
        if download_zip(url, destination):
            return True

    return False


def ckcnn_individual_dir_exists(root: Path) -> bool:
    """Check whether the root has the classification folder we need."""
    possible = [
        root / "dataset" / "individual",
        root / "CK-CNN" / "dataset" / "individual",
        root / "CK-CNN-master" / "dataset" / "individual",
    ]

    for p in possible:
        if p.exists() and any(p.rglob("*")):
            return True

    # Generic deeper search, useful if Kaggle wraps repo folder name.
    for p in root.rglob("individual"):
        if p.is_dir() and (p / "good").exists() and ((p / "broken").exists() or (p / "defective").exists()):
            return True

    return False


def find_public_roots():
    """Find candidate roots. Prioritize CK-CNN because it has dataset/individual class folders."""
    roots = []

    search_roots = [KAGGLE_INPUT_ROOT, RAW_PUBLIC_ROOT, Path("/kaggle/working")]
    for base in search_roots:
        if not base.exists():
            continue

        for p in [base] + [x for x in base.rglob("*") if x.is_dir()]:
            name = p.name.lower()
            full = normalize_text(p)

            # Strong preference: actual classification dataset folder.
            if p.name == "individual" and (p / "good").exists():
                roots.append(p.parent.parent if len(p.parts) >= 2 else p)

            # Repo/folder hints.
            if any(h in full for h in ["ck_cnn", "ck-cnn", "ck_cnnlw", "ck-cnnlw", "corn_kernel", "corn-kernel", "grainset", "efficientmaize", "grainbrain"]):
                roots.append(p)

    # Always include raw public folders if they exist.
    for name in ["CK-CNN", "CK-CNN-master"]:
        p = RAW_PUBLIC_ROOT / name
        if p.exists():
            roots.append(p)

    # De-duplicate while preserving order.
    unique = []
    seen = set()
    for r in roots:
        key = str(r.resolve()) if r.exists() else str(r)
        if key not in seen:
            unique.append(r)
            seen.add(key)

    print("\n[INFO] Candidate dataset roots:")
    for r in unique[:80]:
        print(" ", r)
    if len(unique) > 80:
        print(f"  ... and {len(unique)-80} more")

    return unique


def download_ckcnn_if_needed():
    """Download CK-CNN when no class-labeled image folders are available."""
    ckcnn_dest = RAW_PUBLIC_ROOT / "CK-CNN"
    if ckcnn_dest.exists() and ckcnn_individual_dir_exists(ckcnn_dest):
        print(f"[INFO] CK-CNN classification dataset already available: {ckcnn_dest}")
        return True

    # If an old incomplete CK-CNN folder exists without individual dataset, remove it.
    if ckcnn_dest.exists() and not ckcnn_individual_dir_exists(ckcnn_dest):
        print(f"[WARN] Removing incomplete CK-CNN folder: {ckcnn_dest}")
        shutil.rmtree(ckcnn_dest)

    print("[INFO] Downloading the correct CK-CNN repo with dataset/individual folders...")
    ok = clone_or_download_repo(CK_CNN_GIT_URL, CK_CNN_ZIP_URLS, ckcnn_dest)

    if ok and ckcnn_individual_dir_exists(ckcnn_dest):
        print("[INFO] CK-CNN dataset/individual detected successfully.")
        return True

    print("[WARN] CK-CNN download did not expose dataset/individual.")
    return False


def detect_dataset_name(path: Path) -> str:
    text = normalize_text(path)
    if "ck_cnn" in text or "ck-cnn" in text:
        return "ck_cnn"
    if "grainset" in text or "grain_set" in text or "grain-set" in text:
        return "grainset"
    if "efficientmaize" in text or "efficient_maize" in text or "efficient-maize" in text:
        return "efficientmaize"
    if "grainbrain" in text or "grain_brain" in text or "grain-brain" in text:
        return "grainbrain"
    return "unknown"


def detect_label_from_path(path: Path, dataset_name: str | None = None):
    text = normalize_text(path)

    if is_blocked_non_training_asset(path):
        return None

    # Very important:
    # CK-CNN repo has dataset/individual/good, broken, impurity, rotten folders.
    # Ignore segmentation-style synthesized folders because they are not class-labeled image folders.
    if "/dataset/synthesized/" in text or "/synthesized/" in text:
        return None

    parts = [x.lower().replace("-", "_").replace(" ", "_") for x in path.parts]

    # Exact folder-name mapping has priority.
    if "good" in parts or "healthy" in parts or "normal" in parts:
        return "good"

    if "impurity" in parts or "impurities" in parts:
        return "impurity"

    if "rotten" in parts or "rot" in parts or "mold" in parts or "mildew" in parts or "fungus" in parts:
        return "mold_risk"

    if "broken" in parts or "defective" in parts or "damaged" in parts or "damage" in parts or "defect" in parts:
        return "broken"

    # Fallback text matching for wrapped folder names.
    if "/good/" in text or "_good_" in text or "good_" in text:
        return "good"

    if "/impurity/" in text or "/impurities/" in text or "_impurity_" in text:
        return "impurity"

    if "/rotten/" in text or "_rotten_" in text or "mold" in text or "mildew" in text or "fungus" in text:
        return "mold_risk"

    if "/broken/" in text or "_broken_" in text or "defective" in text or "damaged" in text:
        return "broken"

    return None


def collect_images_from_root(root: Path):
    records = []
    dataset_name = detect_dataset_name(root)

    if not root.exists():
        return records

    image_paths = [p for p in root.rglob("*") if p.is_file() and is_image(p)]
    if not image_paths:
        return records

    print(f"[INFO] Scanning {root} | images found: {len(image_paths)}")

    kept = 0
    examples = []

    for path in image_paths:
        label = detect_label_from_path(path, dataset_name)

        if label not in CLASSES:
            continue

        if not safe_open_image(path):
            continue

        records.append({
            "source_path": str(path),
            "label": label,
            "source_dataset": dataset_name,
            "is_synthetic": False,
        })

        kept += 1
        if len(examples) < 10:
            examples.append((label, str(path)))

    if kept:
        print(f"[INFO] Kept {kept} usable images from {root}")
        print("[INFO] Example mapped images:")
        for label, example_path in examples:
            print(f"  {label:10s} -> {example_path}")

    return records


def collect_public_records():
    # First download the correct CK-CNN if there is not already a usable classification dataset.
    roots_before = find_public_roots()
    initial_records = []
    for root in roots_before:
        initial_records.extend(collect_images_from_root(root))

    # If no usable class folders are found, download CK-CNN, which contains dataset/individual.
    if not initial_records and AUTO_DOWNLOAD_PUBLIC_DATASETS_IF_MISSING:
        print("\n[INFO] No usable class-folder images found yet.")
        print("[INFO] Trying to download CK-CNN classification dataset automatically...")
        download_ckcnn_if_needed()

    roots = find_public_roots()
    records = []
    for root in roots:
        records.extend(collect_images_from_root(root))

    # De-duplicate same path records before hash duplicate removal.
    unique = []
    seen_paths = set()
    for r in records:
        if r["source_path"] not in seen_paths:
            unique.append(r)
            seen_paths.add(r["source_path"])
    records = unique

    if records:
        counts = Counter(r["label"] for r in records)
        print("\n[INFO] Raw collected label counts:")
        print(dict(counts))
        return records

    list_input_folders()
    raise RuntimeError(
        "No usable class-labeled image folders were found. "
        "Fix: run this cell again after Restart Session, or manually add/clone the CK-CNN repo "
        "(https://github.com/vision-cidis/CK-CNN) so dataset/individual/good, broken, impurity, rotten exists."
    )


def remove_duplicate_records(records):
    print("\n[INFO] Removing duplicate images by SHA1 hash...")
    seen = set()
    unique = []

    for r in records:
        path = Path(r["source_path"])

        try:
            h = file_hash(path)
        except Exception:
            continue

        if h in seen:
            continue

        seen.add(h)
        unique.append({**r, "sha1": h})

    print(f"[INFO] Kept {len(unique)} unique images from {len(records)} records.")
    return unique


def limit_per_class(records):
    print("\n[INFO] Limiting max images per class for balanced training...")
    random.seed(SEED)
    output = []

    for label in CLASSES:
        items = [r for r in records if r["label"] == label]
        random.shuffle(items)

        max_n = MAX_PER_CLASS.get(label)
        if max_n is not None:
            items = items[:max_n]

        output.extend(items)
        print(f"  {label:10s}: {len(items)}")

    random.shuffle(output)
    return output


def split_records(records):
    print("\n[INFO] Creating train/val/test split...")
    random.seed(SEED)

    split_items = []

    for label in CLASSES:
        items = [r for r in records if r["label"] == label]
        random.shuffle(items)

        n = len(items)
        if n == 0:
            raise RuntimeError(f"No images found for class: {label}. Check dataset mapping.")

        train_end = int(n * TRAIN_RATIO)
        val_end = train_end + int(n * VAL_RATIO)

        # Ensure every class has test/val examples when possible.
        if n >= 10:
            train = items[:train_end]
            val = items[train_end:val_end]
            test = items[val_end:]
        else:
            train = items[:max(1, n - 2)]
            val = items[max(1, n - 2):max(1, n - 1)]
            test = items[max(1, n - 1):]

        for item in train:
            split_items.append({**item, "split": "train"})
        for item in val:
            split_items.append({**item, "split": "val"})
        for item in test:
            split_items.append({**item, "split": "test"})

        print(f"  {label:10s}: train={len(train)}, val={len(val)}, test={len(test)}")

    random.shuffle(split_items)
    return split_items


def prepare_ready_folders(split_items):
    print("\n[INFO] Preparing ImageFolder-compatible dataset...")
    if READY_ROOT.exists():
        shutil.rmtree(READY_ROOT)

    for split in ["train", "val", "test"]:
        for label in CLASSES:
            (READY_ROOT / split / label).mkdir(parents=True, exist_ok=True)

    final_records = []

    for idx, item in enumerate(split_items):
        src = Path(item["source_path"])
        split = item["split"]
        label = item["label"]
        ext = src.suffix.lower()
        dst = READY_ROOT / split / label / f"{item['source_dataset']}_{label}_{idx:06d}{ext}"

        # Use copy rather than symlink so Kaggle exports/downloads work reliably.
        shutil.copy2(src, dst)

        final_records.append({
            **item,
            "prepared_path": str(dst),
        })

    df = pd.DataFrame(final_records)
    df.to_csv(OUTPUT_DIR / "dataset_manifest.csv", index=False)

    with open(OUTPUT_DIR / "class_names.json", "w") as f:
        json.dump(CLASSES, f, indent=2)

    summary = (
        df.groupby(["split", "label"])
        .size()
        .reset_index(name="images")
        .sort_values(["label", "split"])
    )

    source_summary = (
        df.groupby(["source_dataset", "label"])
        .size()
        .reset_index(name="images")
        .sort_values(["source_dataset", "label"])
    )

    summary.to_csv(OUTPUT_DIR / "dataset_summary_by_split.csv", index=False)
    source_summary.to_csv(OUTPUT_DIR / "dataset_summary_by_source.csv", index=False)

    print("\n[SUMMARY] Split/label counts")
    print(summary.to_string(index=False))
    print("\n[SUMMARY] Source/label counts")
    print(source_summary.to_string(index=False))
    print("\n[INFO] Ready dataset:", READY_ROOT)

    return df


records = collect_public_records()
records = remove_duplicate_records(records)
records = limit_per_class(records)

# Create the template before any split. This keeps a later human review independent of the test set.
for record in records:
    record["review_id"] = review_id_for_path(record["source_path"])
    record["original_label"] = record["label"]
    record["review_decision"] = "not_reviewed"

review_template_df = export_review_template(records)
export_audit_pages(review_template_df)

records, review_decisions_df = apply_human_review(records)
if not records:
    raise RuntimeError("All records were excluded by the review decisions. Check the review CSV.")

reviewed_counts = Counter(record["label"] for record in records)
missing_after_review = [label for label in CLASSES if reviewed_counts.get(label, 0) == 0]
if missing_after_review:
    raise RuntimeError(f"Human review removed all images for class(es): {missing_after_review}. Keep at least one clear class example.")

split_items = split_records(records)
manifest_df = prepare_ready_folders(split_items)

# Save exactly which review decisions were applied for reproducibility.
manifest_df.to_csv(OUTPUT_DIR / "dataset_manifest.csv", index=False)
review_decisions_df.to_csv(OUTPUT_DIR / "review_decisions_used.csv", index=False)
manifest_df.head()


In [ ]:
# ============================================================
# 3.1 Audit summary
# ============================================================

print("Review template rows:", len(review_template_df))
print("Audit image pages:", AUDIT_DIR)
if not review_decisions_df.empty:
    display(review_decisions_df["decision"].value_counts(dropna=False).rename("rows"))
else:
    print("No decision file was applied in this run.")

reviewable_manifest = manifest_df[manifest_df["label"].isin(["broken", "mold_risk"])].copy()
print("Prepared broken/mold rows after review:", len(reviewable_manifest))
display(reviewable_manifest[["split", "label", "original_label", "review_decision", "source_path"]].head(20))


## Data Checks

These cells save the class distribution, source contribution, sample images, image statistics, and the focused broken/mold audit grid. Use these outputs as testing evidence in the final demo.


In [ ]:
# ============================================================
# 4. Data visualization
# ============================================================

# This cell is robust to manifest column names.
# Some notebook versions save the source column as "source_dataset",
# while older plot code expected "dataset". This fixes the KeyError.

print("Manifest columns:", list(manifest_df.columns))

if "dataset" not in manifest_df.columns:
    if "source_dataset" in manifest_df.columns:
        manifest_df["dataset"] = manifest_df["source_dataset"]
    elif "source" in manifest_df.columns:
        manifest_df["dataset"] = manifest_df["source"]
    else:
        manifest_df["dataset"] = "ck_cnn"

if "label" not in manifest_df.columns:
    if "class" in manifest_df.columns:
        manifest_df["label"] = manifest_df["class"]
    elif "target" in manifest_df.columns:
        manifest_df["label"] = manifest_df["target"]
    else:
        raise KeyError("manifest_df must contain a label/class/target column.")

manifest_df["dataset"] = manifest_df["dataset"].fillna("ck_cnn").astype(str)
manifest_df["label"] = manifest_df["label"].astype(str)


def count_imagefolder(root: Path):
    rows = []
    for split in ["train", "val", "test"]:
        for label in CLASSES:
            d = root / split / label
            n = len([p for p in d.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]) if d.exists() else 0
            rows.append({"split": split, "label": label, "images": n})
    return pd.DataFrame(rows)


dist_df = count_imagefolder(READY_ROOT)
display(dist_df)

# Plot 1: train/val/test distribution
pivot = (
    dist_df.pivot(index="label", columns="split", values="images")
    .fillna(0)
    .reindex(CLASSES)
)
for col in ["train", "val", "test"]:
    if col not in pivot.columns:
        pivot[col] = 0

ax = pivot[["train", "val", "test"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Class distribution by split")
ax.set_ylabel("Images")
ax.set_xlabel("Class")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_class_distribution_by_split.png", dpi=180)
plt.show()

# Plot 2: source dataset contribution by class
source_plot_df = (
    manifest_df.groupby(["dataset", "label"])
    .size()
    .reset_index(name="images")
)

display(source_plot_df)

if not source_plot_df.empty:
    source_pivot = (
        source_plot_df.pivot(index="label", columns="dataset", values="images")
        .fillna(0)
        .reindex(CLASSES)
    )
    ax = source_pivot.plot(kind="bar", stacked=True, figsize=(10, 5))
    ax.set_title("Public dataset contribution by class")
    ax.set_ylabel("Images")
    ax.set_xlabel("Class")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "02_dataset_contribution_by_class.png", dpi=180)
    plt.show()
else:
    print("No source contribution plot because manifest_df is empty.")


In [ ]:
# Sample images by class

def show_sample_grid(root=READY_ROOT, split="train", per_class=5):
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(per_class * 2.7, len(CLASSES) * 2.4))

    for row, label in enumerate(CLASSES):
        class_dir = root / split / label
        paths = [p for p in class_dir.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
        random.Random(SEED + row).shuffle(paths)
        sample_paths = paths[:per_class]

        for col in range(per_class):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(sample_paths):
                img = Image.open(sample_paths[col]).convert("RGB")
                ax.imshow(img)
                ax.set_title(label, fontsize=10)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_sample_images_by_class.png", dpi=180)
    plt.show()

show_sample_grid()

In [ ]:
# Image size and RGB summary

def image_stats(root=READY_ROOT, split="train", max_per_class=250):
    rows = []
    for label in CLASSES:
        paths = [p for p in (root / split / label).glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
        random.Random(SEED).shuffle(paths)
        paths = paths[:max_per_class]
        for p in paths:
            try:
                img = Image.open(p).convert("RGB")
                arr = np.asarray(img).astype(np.float32) / 255.0
                rows.append({
                    "label": label,
                    "width": img.width,
                    "height": img.height,
                    "red_mean": arr[:, :, 0].mean(),
                    "green_mean": arr[:, :, 1].mean(),
                    "blue_mean": arr[:, :, 2].mean(),
                })
            except Exception:
                pass
    return pd.DataFrame(rows)

stats_df = image_stats()
display(stats_df.head())

if not stats_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(stats_df["width"], stats_df["height"], alpha=0.35)
    ax.set_title("Image size distribution")
    ax.set_xlabel("Width")
    ax.set_ylabel("Height")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "04_image_size_distribution.png", dpi=180)
    plt.show()

    rgb_summary = stats_df.groupby("label")[["red_mean", "green_mean", "blue_mean"]].mean().reindex(CLASSES)
    display(rgb_summary)

    ax = rgb_summary.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Average RGB channel values by class")
    ax.set_ylabel("Mean channel value")
    ax.set_xlabel("Class")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "05_rgb_summary_by_class.png", dpi=180)
    plt.show()

In [ ]:
# Broken-vs-mold label audit. This is a visual quality check, not an automatic relabelling step.
def show_broken_mold_audit(root=READY_ROOT, split="train", samples_per_class=16):
    labels = ["broken", "mold_risk"]
    fig, axes = plt.subplots(
        len(labels),
        samples_per_class,
        figsize=(samples_per_class * 1.7, len(labels) * 2.2),
    )

    for row, label in enumerate(labels):
        class_dir = root / split / label
        paths = [
            p for p in class_dir.glob("*")
            if p.suffix.lower() in IMAGE_EXTENSIONS
            and "synthetic" not in p.name.lower()
            and "ckcnnlw_synth" not in p.name.lower()
        ]
        random.Random(SEED + row).shuffle(paths)
        selected = paths[:samples_per_class]

        for col in range(samples_per_class):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(selected):
                ax.imshow(Image.open(selected[col]).convert("RGB"))
            if col == 0:
                ax.set_title(label, loc="left", fontsize=12, fontweight="bold")

    plt.suptitle(f"Label audit: broken vs mold_risk ({split})", y=1.02, fontsize=15)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "00_broken_vs_mold_label_audit.png", dpi=180, bbox_inches="tight")
    plt.show()

    summary = pd.DataFrame([
        {
            "split": split,
            "label": label,
            "original_images_reviewed_from": len([
                p for p in (root / split / label).glob("*")
                if p.suffix.lower() in IMAGE_EXTENSIONS
                and "synthetic" not in p.name.lower()
                and "ckcnnlw_synth" not in p.name.lower()
            ]),
        }
        for label in labels
    ])
    summary.to_csv(OUTPUT_DIR / "broken_mold_label_audit_summary.csv", index=False)
    display(summary)


show_broken_mold_audit()


## Model and Training

The model is MobileNetV3 Large with ImageNet transfer learning. The classifier head is replaced with four MaizeGuard outputs. Training uses a head phase followed by fine-tuning. The selected checkpoint is chosen from validation macro F1, using validation broken/mold F1 as the tie-breaker.


In [ ]:
# ============================================================
# 5. Dataset, preprocessing, and dataloaders
# ============================================================

# Conservative augmentation. Hue and saturation are intentionally not changed,
# because color/discoloration can be part of the visible mold-risk signal.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(degrees=7, fill=(245, 245, 240)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


class MaizeImageDataset(Dataset):
    def __init__(self, root: Path, split: str, transform=None):
        self.root = root
        self.split = split
        self.transform = transform
        self.samples = []

        for label in CLASSES:
            class_dir = root / split / label
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.glob("*")):
                if path.suffix.lower() in IMAGE_EXTENSIONS:
                    self.samples.append((path, CLASS_TO_IDX[label]))

        if not self.samples:
            raise RuntimeError(f"No images found in {root}/{split}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, target, str(path)


train_ds = MaizeImageDataset(READY_ROOT, "train", train_transform)
val_ds = MaizeImageDataset(READY_ROOT, "val", eval_transform)
test_ds = MaizeImageDataset(READY_ROOT, "test", eval_transform)


def split_counts(dataset):
    counts = Counter(target for _, target in dataset.samples)
    return {IDX_TO_CLASS[index]: counts.get(index, 0) for index in range(len(CLASSES))}


train_counts = split_counts(train_ds)
print("Train:", len(train_ds), train_counts)
print("Validation:", len(val_ds), split_counts(val_ds))
print("Test:", len(test_ds), split_counts(test_ds))

# Gentle class weights help minority classes without repeatedly duplicating the same examples.
counts_array = np.array([train_counts[label] for label in CLASSES], dtype=np.float32)
if np.any(counts_array <= 0):
    raise RuntimeError(f"A training class has zero images: {train_counts}")

raw_weights = (counts_array.max() / counts_array) ** CLASS_WEIGHT_POWER
normalised_weights = raw_weights / raw_weights.mean()
CLASS_WEIGHTS = torch.tensor(normalised_weights, dtype=torch.float32, device=DEVICE)
print("Training class weights:", dict(zip(CLASSES, np.round(normalised_weights, 3))))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)


In [ ]:
# ============================================================
# 6. Model architecture
# ============================================================

def build_model(model_name=MODEL_NAME, num_classes=len(CLASSES)):
    try:
        return timm.create_model(
            model_name,
            pretrained=True,
            num_classes=num_classes,
        )
    except Exception as error:
        raise RuntimeError(
            "Could not load ImageNet pretrained weights through timm. "
            "On Kaggle, turn Internet on for the first run or attach a dataset/cache that contains the weights."
        ) from error


model = build_model().to(DEVICE)
architecture_text = str(model)
(OUTPUT_DIR / "model_architecture_full.txt").write_text(architecture_text, encoding="utf-8")

print(f"Model created on device: {DEVICE}")
print("Classifier layer:")
print(model.get_classifier() if hasattr(model, "get_classifier") else "Classifier summary unavailable")

num_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
arch_summary = pd.DataFrame([
    {"item": "Backbone", "value": MODEL_NAME},
    {"item": "Pretraining", "value": "ImageNet via timm"},
    {"item": "Input image size", "value": IMG_SIZE},
    {"item": "Classes", "value": ", ".join(CLASSES)},
    {"item": "Optimizer", "value": "AdamW"},
    {"item": "Training loss", "value": "Gentle class-weighted CrossEntropyLoss"},
    {"item": "Sampling", "value": "Shuffle training loader; no weighted sampler"},
    {"item": "Human label audit", "value": "Optional decision file before splitting"},
    {"item": "Device used", "value": str(DEVICE)},
    {"item": "CUDA usable", "value": CUDA_USABLE},
    {"item": "Total parameters", "value": num_params},
    {"item": "Trainable parameters", "value": trainable_params},
])
display(arch_summary)
arch_summary.to_csv(OUTPUT_DIR / "model_architecture_summary.csv", index=False)


In [ ]:
# ============================================================
# 7. Training functions
# ============================================================

BROKEN_IDX = CLASS_TO_IDX["broken"]
MOLD_IDX = CLASS_TO_IDX["mold_risk"]


def set_trainable(model, trainable: bool):
    for parameter in model.parameters():
        parameter.requires_grad = trainable


def training_criterion():
    weights = CLASS_WEIGHTS if USE_CLASS_WEIGHTED_LOSS else None
    return nn.CrossEntropyLoss(weight=weights)


@torch.no_grad()
def evaluate_basic(model, loader, criterion):
    model.eval()
    losses, ys, preds = [], [], []

    for x, y, _ in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(float(loss.item()))
        ys.extend(y.cpu().numpy().tolist())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    ys_array = np.asarray(ys)
    preds_array = np.asarray(preds)
    accuracy = accuracy_score(ys_array, preds_array)
    _, _, macro_f1, _ = precision_recall_fscore_support(
        ys_array, preds_array, average="macro", zero_division=0
    )

    pair_mask = np.isin(ys_array, [BROKEN_IDX, MOLD_IDX])
    if pair_mask.any():
        _, _, broken_mold_f1, _ = precision_recall_fscore_support(
            ys_array[pair_mask],
            preds_array[pair_mask],
            labels=[BROKEN_IDX, MOLD_IDX],
            average="macro",
            zero_division=0,
        )
    else:
        broken_mold_f1 = float("nan")

    return float(np.mean(losses)), float(accuracy), float(macro_f1), float(broken_mold_f1)


def train_one_phase(model, phase_name, train_loader, val_loader, epochs, lr, train_backbone):
    if train_backbone:
        set_trainable(model, True)
    else:
        set_trainable(model, False)
        for name, parameter in model.named_parameters():
            if any(token in name.lower() for token in ["classifier", "head", "fc"]):
                parameter.requires_grad = True

    optimizer = torch.optim.AdamW(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    train_criterion = training_criterion()
    eval_criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))

    use_amp = DEVICE.type == "cuda" and CUDA_USABLE
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None

    trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"[INFO] Phase={phase_name} | trainable parameters={trainable_count:,} | device={DEVICE}")

    best_state = None
    best_info = {
        "phase": phase_name,
        "epoch": None,
        "val_macro_f1": -1.0,
        "val_broken_mold_f1": -1.0,
    }
    no_improvement = 0
    phase_logs = []

    for epoch in range(1, epochs + 1):
        started = time.time()
        model.train()
        losses = []

        for x, y, _ in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            if use_amp and scaler is not None:
                with torch.cuda.amp.autocast(enabled=True):
                    logits = model(x)
                    loss = train_criterion(logits, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss = train_criterion(logits, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            losses.append(float(loss.item()))

        scheduler.step()
        val_loss, val_accuracy, val_macro_f1, val_broken_mold_f1 = evaluate_basic(
            model, val_loader, eval_criterion
        )
        log = {
            "phase": phase_name,
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": float(np.mean(losses)),
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_macro_f1": val_macro_f1,
            "val_broken_mold_f1": val_broken_mold_f1,
            "seconds": round(time.time() - started, 2),
            "device": str(DEVICE),
        }
        phase_logs.append(log)
        print(log)

        better_macro = val_macro_f1 > best_info["val_macro_f1"] + 1e-8
        tied_better_pair = (
            np.isclose(val_macro_f1, best_info["val_macro_f1"])
            and val_broken_mold_f1 > best_info["val_broken_mold_f1"] + 1e-8
        )
        if better_macro or tied_better_pair:
            best_info = {
                "phase": phase_name,
                "epoch": epoch,
                "val_macro_f1": float(val_macro_f1),
                "val_broken_mold_f1": float(val_broken_mold_f1),
            }
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            no_improvement = 0
        else:
            no_improvement += 1
            if no_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"[INFO] Early stopping {phase_name} at epoch {epoch}.")
                break

    return phase_logs, best_state, best_info


model = build_model().to(DEVICE)
training_logs = []

try:
    head_logs, head_best_state, head_best_info = train_one_phase(
        model, "head", train_loader, val_loader, HEAD_EPOCHS, LR_HEAD, train_backbone=False
    )
    training_logs.extend(head_logs)
    if head_best_state is not None:
        model.load_state_dict(head_best_state)

    finetune_logs, finetune_best_state, finetune_best_info = train_one_phase(
        model, "finetune", train_loader, val_loader, FINETUNE_EPOCHS, LR_FINETUNE, train_backbone=True
    )
    training_logs.extend(finetune_logs)
except RuntimeError as error:
    error_text = str(error)
    if "no kernel image is available" in error_text or "cudaErrorNoKernelImageForDevice" in error_text:
        raise RuntimeError(
            "The selected GPU is incompatible with this PyTorch build. Choose T4/V100/A100 or let V9 use CPU."
        ) from error
    raise

candidates = [(head_best_state, head_best_info), (finetune_best_state, finetune_best_info)]
candidates = [(state, info) for state, info in candidates if state is not None]
if not candidates:
    raise RuntimeError("Training did not produce a usable checkpoint.")

selected_state, selected_info = max(
    candidates,
    key=lambda item: (item[1]["val_macro_f1"], item[1]["val_broken_mold_f1"]),
)
model.load_state_dict(selected_state)
print("[INFO] Checkpoint selected from validation metrics:", selected_info)

logs_df = pd.DataFrame(training_logs)
display(logs_df)
logs_df.to_csv(OUTPUT_DIR / "training_logs.csv", index=False)

CONFIG["selected_training_phase"] = selected_info["phase"]
CONFIG["selected_epoch"] = selected_info["epoch"]
CONFIG["selected_validation_macro_f1"] = selected_info["val_macro_f1"]
CONFIG["selected_validation_broken_mold_f1"] = selected_info["val_broken_mold_f1"]
(OUTPUT_DIR / "training_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

checkpoint_payload = {
    "model_name": MODEL_NAME,
    "state_dict": model.state_dict(),
    "classes": CLASSES,
    "img_size": IMG_SIZE,
    "config": CONFIG,
}

best_model_path = OUTPUT_DIR / f"{MODEL_NAME}_best.pt"
public_model_path = OUTPUT_DIR / "maizeguard_public_best_model.pt"
torch.save(checkpoint_payload, best_model_path)
torch.save(checkpoint_payload, public_model_path)

metadata = {
    **CONFIG,
    "class_names": CLASSES,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "model_file": public_model_path.name,
}
(OUTPUT_DIR / "maizeguard_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved model:", public_model_path)
print("Saved metadata:", OUTPUT_DIR / "maizeguard_model_metadata.json")


In [ ]:
# ============================================================
# 8. Training visualization
# ============================================================

if not logs_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    for phase in logs_df["phase"].unique():
        sub = logs_df[logs_df["phase"] == phase]
        ax.plot(sub["epoch"], sub["train_loss"], marker="o", label=f"{phase} train loss")
        ax.plot(sub["epoch"], sub["val_loss"], marker="s", label=f"{phase} val loss")

    ax.set_title("Training and validation loss")
    ax.set_xlabel("Epoch within phase")
    ax.set_ylabel("Loss")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "06_training_validation_loss.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    for phase in logs_df["phase"].unique():
        sub = logs_df[logs_df["phase"] == phase]
        ax.plot(sub["epoch"], sub["val_accuracy"], marker="o", label=f"{phase} val accuracy")
        ax.plot(sub["epoch"], sub["val_macro_f1"], marker="s", label=f"{phase} val macro F1")
        ax.plot(sub["epoch"], sub["val_broken_mold_f1"], marker="^", linestyle="--", label=f"{phase} broken/mold F1")

    ax.set_title("Validation accuracy, macro F1, and broken/mold F1")
    ax.set_xlabel("Epoch within phase")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "07_validation_accuracy_macro_f1.png", dpi=180)
    plt.show()


## Evaluation and Export

Evaluation uses raw argmax predictions on the untouched test split. The checkpoint and metadata are saved with the filenames expected by the backend.


In [ ]:
# ============================================================
# 9. Official evaluation: raw argmax only
# ============================================================

@torch.no_grad()
def predict_loader_raw(model, loader):
    model.eval()
    y_true, y_pred, y_prob, paths = [], [], [], []

    for x, y, batch_paths in loader:
        x = x.to(DEVICE)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        y_true.extend(y.numpy().tolist())
        y_pred.extend(preds.tolist())
        y_prob.extend(probs.tolist())
        paths.extend(list(batch_paths))

    return np.array(y_true), np.array(y_pred), np.array(y_prob), paths


y_true, y_pred, y_prob, test_paths = predict_loader_raw(model, test_loader)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

pair_test_mask = np.isin(y_true, [BROKEN_IDX, MOLD_IDX])
_, _, broken_mold_test_f1, _ = precision_recall_fscore_support(
    y_true[pair_test_mask],
    y_pred[pair_test_mask],
    labels=[BROKEN_IDX, MOLD_IDX],
    average="macro",
    zero_division=0,
)

print("Final evaluation mode:", OFFICIAL_EVAL_MODE)
print(f"Accuracy: {acc:.4f}")
print(f"Macro precision: {precision_macro:.4f}")
print(f"Macro recall: {recall_macro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print(f"Broken/mold test F1: {broken_mold_test_f1:.4f}")

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=CLASSES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df)
report_df.to_csv(OUTPUT_DIR / "classification_report_raw_argmax.csv")

metrics_summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "device": str(DEVICE),
    "evaluation_mode": OFFICIAL_EVAL_MODE,
    "accuracy": acc,
    "macro_precision": precision_macro,
    "macro_recall": recall_macro,
    "macro_f1": f1_macro,
    "broken_mold_test_f1": broken_mold_test_f1,
    "test_samples": len(y_true),
    "mistakes": int((y_true != y_pred).sum()),
    "selected_validation_phase": selected_info["phase"],
    "selected_validation_macro_f1": selected_info["val_macro_f1"],
    "selected_validation_broken_mold_f1": selected_info["val_broken_mold_f1"],
    "model_path": str(best_model_path),
}])
display(metrics_summary)
metrics_summary.to_csv(OUTPUT_DIR / "model_metrics_summary.csv", index=False)

records = []
for true_i, pred_i, probs, path in zip(y_true, y_pred, y_prob, test_paths):
    sorted_probs = np.sort(probs)[::-1]
    margin = float(sorted_probs[0] - sorted_probs[1]) if len(sorted_probs) > 1 else float(sorted_probs[0])
    row = {
        "path": path,
        "true_label": IDX_TO_CLASS[int(true_i)],
        "pred_label": IDX_TO_CLASS[int(pred_i)],
        "confidence": float(probs[pred_i]),
        "top2_margin": margin,
        "broken_mold_probability_gap": float(abs(probs[BROKEN_IDX] - probs[MOLD_IDX])),
        "correct": bool(true_i == pred_i),
    }
    for idx, cls in IDX_TO_CLASS.items():
        row[f"prob_{cls}"] = float(probs[idx])
    records.append(row)

pred_df = pd.DataFrame(records)
pred_df.to_csv(OUTPUT_DIR / "test_predictions_and_errors_raw_argmax.csv", index=False)
display(pred_df.head())

mistakes_df = pred_df[pred_df["correct"] == False]
display(mistakes_df)
print(f"Mistakes: {len(mistakes_df)} / {len(pred_df)}")


In [ ]:
# ============================================================
# 10. Confusion matrices and per-class metrics
# ============================================================

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm)
ax.set_title(f"Confusion Matrix — {MODEL_NAME} (raw argmax)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_confusion_matrix_raw_argmax.png", dpi=180)
plt.show()

cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, vmin=0, vmax=1)
ax.set_title("Normalized Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="black")

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "09_normalized_confusion_matrix.png", dpi=180)
plt.show()

per_class = report_df.loc[CLASSES, ["precision", "recall", "f1-score", "support"]].reset_index().rename(columns={"index": "class"})
display(per_class)
per_class.to_csv(OUTPUT_DIR / "per_class_metrics.csv", index=False)

ax = per_class.set_index("class")[["precision", "recall", "f1-score"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Per-class precision, recall, and F1-score")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "10_per_class_metrics.png", dpi=180)
plt.show()

# Focused diagnostic for the two classes that this experiment aims to separate.
pair_truth_mask = np.isin(y_true, [BROKEN_IDX, MOLD_IDX])
pair_cm = confusion_matrix(y_true[pair_truth_mask], y_pred[pair_truth_mask], labels=[BROKEN_IDX, MOLD_IDX])
pair_cm_df = pd.DataFrame(pair_cm, index=["true_broken", "true_mold_risk"], columns=["pred_broken", "pred_mold_risk"])
display(pair_cm_df)
pair_cm_df.to_csv(OUTPUT_DIR / "broken_mold_pair_confusion.csv")

pair_rows = pred_df[pred_df["true_label"].isin(["broken", "mold_risk"])].copy()
pair_rows.to_csv(OUTPUT_DIR / "broken_mold_test_predictions.csv", index=False)
print("Saved focused broken/mold reports for comparison with the prior notebook.")


In [ ]:
# ============================================================
# 11. Mistake visualization and confidence analysis
# ============================================================

def show_mistakes(mistakes_df, max_images=8):
    if mistakes_df.empty:
        print("No mistakes found on the test set.")
        return

    sample = mistakes_df.head(max_images)
    cols = min(4, len(sample))
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(
            f"true: {row['true_label']}\npred: {row['pred_label']}\nconf: {row['confidence']:.2f}",
            fontsize=10,
        )

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "11_mistakes_raw_argmax.png", dpi=180)
    plt.show()

show_mistakes(mistakes_df)

confidence_summary = (
    pred_df.groupby("true_label")
    .agg(
        samples=("true_label", "count"),
        accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        mean_top2_margin=("top2_margin", "mean"),
    )
    .reset_index()
)
display(confidence_summary)
confidence_summary.to_csv(OUTPUT_DIR / "confidence_summary_by_class.csv", index=False)

if not mistakes_df.empty:
    mistake_pairs = mistakes_df.groupby(["true_label", "pred_label"]).size().reset_index(name="mistakes")
    display(mistake_pairs)
    mistake_pairs.to_csv(OUTPUT_DIR / "mistake_pairs.csv", index=False)

## Deployment Safety Helper

This helper mirrors the backend response format. It returns `needs_review` for poor image quality, low confidence, small top-two margin, close broken/mold probability, or inconsistent image views.


In [ ]:
# ============================================================
# 12. Deployment-safe prediction helpers
# ============================================================

RECOMMENDATIONS = {
    "good": {"risk": "Low", "action": "Store safely or prepare for sale", "recommendation": "The maize appears clean. Store in a dry place and monitor normally."},
    "broken": {"risk": "Medium", "action": "Sort before storage", "recommendation": "Remove visibly broken or damaged kernels before storage or sale."},
    "impurity": {"risk": "Medium", "action": "Clean and re-screen", "recommendation": "Remove foreign materials such as stones, husks, dust, or debris."},
    "mold_risk": {"risk": "High", "action": "Do not store - refer for checking", "recommendation": "Possible visible mold-risk evidence needs careful checking before storage or consumption."},
    "needs_review": {"risk": "Unclear", "action": "Needs review", "recommendation": "Capture a clearer, close-up photo of shelled maize on a plain surface."},
}


def broken_mold_review_reason(probabilities: np.ndarray):
    broken_probability = float(probabilities[BROKEN_IDX])
    mold_probability = float(probabilities[MOLD_IDX])
    gap = abs(broken_probability - mold_probability)
    if gap < BROKEN_MOLD_DEPLOYMENT_MARGIN_BELOW and min(broken_probability, mold_probability) >= BROKEN_MOLD_DEPLOYMENT_MIN_PROBABILITY:
        return "Broken-kernel and mold-risk scores are too close to separate reliably."
    return None


def image_quality_reason(image: Image.Image):
    width, height = image.size
    if min(width, height) < MIN_EXTERNAL_IMAGE_SIDE:
        return f"Image is too small ({width}x{height}); upload a clearer image."

    small = image.convert("RGB").resize((96, 96))
    array = np.asarray(small, dtype=np.float32)
    brightness = array.mean(axis=2)
    if float((brightness > 242).mean()) > 0.92:
        return "Image is mostly blank or over-exposed."
    if float((brightness < 25).mean()) > 0.80:
        return "Image is too dark."

    gray = np.asarray(image.convert("L").resize((256, 256)), dtype=np.float32)
    laplacian = -4 * gray + np.roll(gray, 1, 0) + np.roll(gray, -1, 0) + np.roll(gray, 1, 1) + np.roll(gray, -1, 1)
    if float(np.var(laplacian)) < MIN_EXTERNAL_BLUR_SCORE:
        return "Image is too blurry for reliable screening."
    return None


def make_prediction_views(image: Image.Image):
    rgb = image.convert("RGB")
    width, height = rgb.size
    views = [("full", rgb)]
    if min(width, height) < 360:
        return views

    side = int(min(width, height) * 0.62)
    boxes = {
        "center": ((width - side) // 2, (height - side) // 2),
        "top_left": (0, 0),
        "top_right": (width - side, 0),
        "bottom_left": (0, height - side),
        "bottom_right": (width - side, height - side),
    }
    for name, (left, top) in boxes.items():
        right = min(left + side, width)
        bottom = min(top + side, height)
        left = max(0, right - side)
        top = max(0, bottom - side)
        if right > left and bottom > top:
            views.append((name, rgb.crop((left, top, right, bottom))))
    return views


@torch.no_grad()
def predict_image_for_deployment(image_path: Path):
    model.eval()
    original = Image.open(image_path).convert("RGB")
    quality_reason = image_quality_reason(original)
    views = make_prediction_views(original)

    all_probabilities = []
    view_predictions = []
    for name, view in views:
        x = eval_transform(view).unsqueeze(0).to(DEVICE)
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        all_probabilities.append(probs)
        best_index = int(np.argmax(probs))
        view_predictions.append({"view": name, "label": IDX_TO_CLASS[best_index], "confidence": round(float(probs[best_index]), 4)})

    # Every view contributes equally.
    probabilities = np.stack(all_probabilities, axis=0).mean(axis=0)
    probabilities = probabilities / probabilities.sum()
    raw_index = int(np.argmax(probabilities))
    raw_label = IDX_TO_CLASS[raw_index]
    confidence = float(probabilities[raw_index])
    ordered = np.sort(probabilities)[::-1]
    top2_margin = float(ordered[0] - ordered[1])

    vote_count = sum(item["label"] == raw_label for item in view_predictions)
    patch_consensus = vote_count / max(len(view_predictions), 1)
    pair_reason = broken_mold_review_reason(probabilities)
    review_reason = quality_reason or pair_reason
    if review_reason is None and patch_consensus < PATCH_CONSENSUS_MINIMUM:
        review_reason = "Different image regions gave inconsistent predictions."

    needs_review = (
        review_reason is not None
        or confidence < NEEDS_REVIEW_CONFIDENCE_BELOW
        or top2_margin < NEEDS_REVIEW_TOP2_MARGIN_BELOW
    )
    final_label = "needs_review" if needs_review else raw_label

    return {
        "label": final_label,
        "raw_label": raw_label,
        "confidence": round(confidence, 4),
        "confidence_percent": round(confidence * 100, 2),
        "top2_margin": round(top2_margin, 4),
        "broken_mold_probability_gap": round(abs(float(probabilities[BROKEN_IDX]) - float(probabilities[MOLD_IDX])), 4),
        "needs_review": bool(needs_review),
        "review_reason": review_reason,
        "view_count": len(view_predictions),
        "patch_consensus": round(float(patch_consensus), 4),
        "view_predictions": view_predictions,
        "probabilities": {label: round(float(probabilities[index]), 4) for index, label in IDX_TO_CLASS.items()},
        **RECOMMENDATIONS[final_label],
    }


example_path = Path(test_paths[0])
print("Example image:", example_path)
predict_image_for_deployment(example_path)


## Final Run Summary

Use `v9_final_decision_summary.json` and the exported metrics files for the submission video and README. After the final Kaggle run, copy `maizeguard_public_best_model.pt`, `maizeguard_model_metadata.json`, and `class_names.json` into `model_server/model_exports/`.


In [ ]:
# ============================================================
# 15. Final V9 decision summary
# ============================================================

final_decision_summary = {
    "selected_validation_macro_f1": selected_info["val_macro_f1"],
    "selected_validation_broken_mold_f1": selected_info["val_broken_mold_f1"],
    "official_test_macro_f1": float(f1_macro),
    "official_test_broken_mold_f1": float(broken_mold_test_f1),
    "official_test_accuracy": float(acc),
    "human_review_file_applied": not review_decisions_df.empty,
    "review_template": str(OUTPUT_DIR / "broken_mold_review_template.csv"),
    "audit_pages": str(AUDIT_DIR),
    "next_action": "Review ambiguous broken/mold source images, save broken_mold_review_decisions.csv, and run the notebook again." if review_decisions_df.empty else "Use this reviewed-label run for backend export if validation macro F1 and broken/mold F1 meet the project objective.",
}

print(json.dumps(final_decision_summary, indent=2))
(OUTPUT_DIR / "v9_final_decision_summary.json").write_text(json.dumps(final_decision_summary, indent=2), encoding="utf-8")
